# Reproducing Asari and Zador (2009) on CRCNS ac-1

Paper: H. Asari and A. M. Zador (2009), "Long-Lasting Context Dependence Constrains Neural Encoding
Models in Rodent Auditory Cortex", J Neurophysiol 102:2638-2656, doi:10.1152/jn.00577.2009.

Dataset: CRCNS ac-1 (doi:10.6080/K0KW5CXR), "Auditory cortex and thalamic neuronal responses to
various natural and synthetic sounds", read anonymously from the public S3 bucket `crcnsarchive`
(`ac-1/`), streaming the zip files by range requests.

Version. CRCNS data sets are not versioned, so the version is recorded as `unversioned`; the
cell below prints the bucket listing and the MD5 sums of `checksums.md5` for the three zip files
read, so that a later change can be detected. There is only one published version to choose; the
paper cites the data set without a version.

Claims file: `claims/asari-2009.yaml`, written from the version of record (text extraction and
PDF), without the online supplement. This notebook is a repeat attempt, made blind to the earlier
ones.

The data set holds, besides the recordings, the authors' MATLAB analysis code
(`asari/Tools/CxtDep.m` and others in `crcns-ac1.zip`). Method details the claims file leaves open
are taken from the paper first and, where the paper is silent, from that code; each is cited in
the helper module `ctxdep.py` and below.

| Claim | Tested here |
|---|---|
| methods-recording-counts | tested (voltage-clamp counts: no data) |
| a1-mgb-firing-rates | tested |
| fig2-example-reliability | tested |
| fig2-example-long-context | tested (example identified by its reliability) |
| fig3a-proportion-significant | tested |
| fig3a-significance-decay | tested |
| fig3b-fractional-power-decay | tested |
| timescale-longer-than-membrane | tested |
| fig4-predictability-upper-bound | tested |
| fig4-strf-performance-low | tested in `strf_asari_2009.ipynb` |
| fig5-example-conductances | blocked, no voltage-clamp recordings |
| fig6-conductance-decay | blocked, no voltage-clamp recordings |
| fig8a-proportion-by-property | tested (property sets rebuilt from the stimulus parameters) |
| fig8b-decay-by-property | tested (as fig8a) |
| fig8a-significance-fits | tested (as fig8a) |
| fig8c-relative-effects | tested (as fig8a) |
| fig8-natural-synthetic-probes-alike | tested (as fig8a) |
| fig9-mgb-fast-decay | tested |
| fig9-example-first-bin | tested |
| fig7-example-frequency-context | blocked, no recording with the example's design (search shown) |
| figs1-silence-probe | tested |

In [ ]:
import os, time, zlib, resource, collections
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import ndimage, stats
import ctxdep as cd

T0 = time.time()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)


def seed(*parts):
    # one generator per probe and reading, independent of processing order
    return np.random.default_rng([zlib.crc32("/".join(map(str, parts)).encode())])

## Data: listing, checksums, and one pass over the recordings

In [ ]:
listing = cd.bucket_listing()
md5 = cd.checksums()
print("bucket listing of ac-1/:")
for k, s in listing:
    print(f"  {k:45s} {int(s):>12d} bytes")
print("MD5 (checksums.md5) of the files read:")
for z in cd.ZIPS:
    print(f"  {z:35s} {md5[z]}")
S = cd.summary("asari")
for g in S:
    print(f"{len(g['cells']):3d} cells | {g['description']} | {g['reference'][:60]}")
W = cd.summary("wehr")
for g in W:
    print(f"{len(g['cells']):3d} cells | Wehr | {g['description']}")

In [ ]:
NAT, MGB, SYN, MISC = S[0], S[1], S[2], S[3]
assert "natural sound" in NAT["description"] and "Thalamic" in MGB["description"]
assert "synthetic" in SYN["description"]


def fig8_tests(rec):
    """Significance (contexts of one property set only) and fractional power of every probe
    with at least two contexts of a set, each with at least 4 trials (Fig. 8)."""
    kinds = [cd.stim_kind(s) for s in rec.stimuli]
    sets = cd.property_sets(rec)
    tests = []
    for p in (cd.build_probes(rec, include_silence_context=False) if sets else []):
        for k, S in enumerate(sets):
            ctx = [c for c, X in zip(p.contexts, p.traces) if c in S["members"] and X.shape[0] >= 4]
            if len(ctx) < 2:
                continue
            r = cd.kw_significance(p, contexts_subset=ctx, rng=seed(rec.path, p.probe, "fig8", k))
            fp = cd.frac_power(p, contexts_subset=ctx)
            tests.append(dict(cell=p.cell, path=rec.path, probe=p.probe, kind=kinds[p.probe - 1], prop=S["prop"], label=S["label"],
                              members=tuple(S["members"]), main=S["main"], own=p.probe in S["members"], ctx=tuple(ctx), rep=r["rep"],
                              dur=p.duration, sig=r["sig"], noise=r["noise"].astype(np.float32),
                              **{k_: v.astype(np.float32) for k_, v in fp.items()}))
    return dict(sets=[(S["prop"], S["label"], tuple(S["members"]), S["main"]) for S in sets], tests=tests)


def proc_a1(job):
    group, cell, path, desc = job
    rec = cd.load_recording(path, keep_raw=True)
    out = dict(group=group, cell=cell, path=path, desc=desc, type=rec.rec_type, sf=rec.sf, unit=rec.unit,
               dur=rec.n_samples / rec.sf, rep=rec.rep, nfrag=len(rec.durations), trig=collections.Counter(t[0] for t in rec.triggers),
               **rec.meta)
    out["rates"] = {h: cd.window_rates(rec, cd.a1_spikes(rec, h)) for h in (10, 15, 20)}
    rec.raw = None
    if desc != "tuning curve":
        out["clock"] = cd.onset_contrast(rec)
        out["counts"] = {inc: [(p.probe, tuple(p.contexts), tuple(p.n_trials), p.duration) for p in cd.build_probes(rec, include_silence_context=inc)]
                         for inc in (True, False)}
        out["stim"] = [(str(getattr(s, "name", "")), str(getattr(s, "description", "")),
                        np.atleast_1d(getattr(s.param, "ff", [])).tolist(), cd.stim_kind(s), cd.stim_bands(s)) for s in rec.stimuli]
    if group == "synthetic" and desc != "tuning curve":
        out["fig8"] = fig8_tests(rec)
    if group == "natural" and desc != "tuning curve":
        res = []
        for inc in (True, False):
            for p in cd.build_probes(rec, include_silence_context=inc, include_silence_probe=inc):
                r = cd.kw_significance(p, rng=seed(path, p.probe, inc))
                fp = cd.frac_power(p)
                d = dict(cell=cell, path=path, desc=desc, probe=p.probe, ctx=tuple(p.contexts), ntr=tuple(p.n_trials),
                         dur=p.duration, silence=inc, rep=r["rep"], sig=r["sig"], noise=r["noise"], pval=r["p"], **fp)
                if inc:
                    d["traces"] = p.traces
                res.append(d)
        out["probes"] = res
        out["spont_ac"] = {0.0: cd.spont_autocorr(rec, skip=0.0), 1.0: cd.spont_autocorr(rec, skip=1.0)}
    return out


def proc_mgb(job):
    group, cell, path, desc = job
    rec = cd.load_recording(path, keep_raw=True, filtered=False)
    info = {}
    sp = cd.extract_spikes(rec.raw, info=info)
    # second detector (a reading): half the automatic threshold, to count spikes it may miss
    sp5 = cd.extract_spikes(rec.raw, threshold=info["threshold"] / 2)
    out = dict(group=group, cell=cell, path=path, desc=desc, type=rec.rec_type, sf=rec.sf, unit=rec.unit,
               dur=rec.n_samples / rec.sf, rep=rec.rep, nspk=len(sp), nspk5=len(sp5), **info, **rec.meta)
    out["rates"] = cd.window_rates(rec, sp)
    out["rates5"] = cd.window_rates(rec, sp5)
    rec.raw = None
    if desc != "tuning curve":
        segs = [s for s in cd.sequence_segments(rec) if s["seq"] is not None]
        post = sum(((sp >= s["on"]) & (sp < s["on"] + cd.SF // 2)).sum() for s in segs) / (0.5 * len(segs))
        pre = sum(((sp >= s["on"] - cd.SF // 2) & (sp < s["on"])).sum() for s in segs) / (0.5 * len(segs))
        out["clock"] = dict(post=post, pre=pre)
        res = []
        for inc in (True, False):
            for p, p5 in zip(cd.build_probes(rec, include_silence_context=inc, spikes=sp),
                             cd.build_probes(rec, include_silence_context=inc, spikes=sp5)):
                r = cd.psth_sd(p)
                res.append(dict(cell=cell, path=path, desc=desc, probe=p.probe, ctx=tuple(p.contexts), ntr=tuple(p.n_trials),
                                dur=p.duration, silence=inc, rep=r["rep"], sd=r["sd"], psth=r["psth"], sd5=cd.psth_sd(p5)["sd"]))
        out["probes"] = res
    return out


jobs_a1 = [(gname, c["cell"], f, d) for gname, g in (("natural", NAT), ("synthetic", SYN), ("various", MISC))
           for c in g["cells"] for f, d in zip(c["files"], c["descriptions"])]
jobs_mgb = [("mgb", c["cell"], f, d) for c in MGB["cells"] for f, d in zip(c["files"], c["descriptions"])]
print(len(jobs_a1), "A1 files;", len(jobs_mgb), "MGB files")
with ThreadPoolExecutor(4) as ex:
    A1 = list(ex.map(proc_a1, jobs_a1))
    MG = list(ex.map(proc_mgb, jobs_mgb))
print(f"loaded in {time.time() - T0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

In [ ]:
F = pd.DataFrame([{k: v for k, v in o.items() if k in ("group", "cell", "path", "desc", "type", "sf", "unit", "dur", "rep",
                                                    "age", "vmin", "vmax", "vmedian")} for o in A1 + MG])
print("files by group and recording type:")
print(F.groupby(["group", "type"]).agg(files=("path", "size"), cells=("cell", "nunique"), animals=("cell", lambda c: c.str[:12].nunique())))
print("\nsampling rates:", sorted(F.sf.unique()), "| units:", sorted(F.unit.unique()))
print("recording types in the whole data set:", sorted(F.type.unique()), "(no voltage-clamp recording: every trace is in mV)")
print("\nmedian membrane potential of whole-cell files (mV): range %.1f to %.1f; trace extremes %.1f to %.1f mV"
      % (F[F.type == "Whole-cell"].vmedian.min(), F[F.type == "Whole-cell"].vmedian.max(),
         F[F.type == "Whole-cell"].vmin.min(), F[F.type == "Whole-cell"].vmax.max()))
print("recording length per file (s): %.0f to %.0f" % (F.dur.min(), F.dur.max()))
ages = F.drop_duplicates("cell")
print("animal age (days) as stored, cells per age:", dict(sorted(collections.Counter(ages.age).items())))
print("cells outside the paper's 20 to 28 days or without an age:",
      ages[~ages.age.isin([str(a) for a in range(20, 29)])][["group", "cell", "age"]].values.tolist())
print("\nnatural sound A1 recordings by design:")
print(F[(F.group == "natural") & (F.desc != "tuning curve")].groupby("desc").agg(files=("path", "size"), cells=("cell", "nunique"), rep_min=("rep", "min"), rep_max=("rep", "max")))
print("\nMGB recordings by design:")
print(F[(F.group == "mgb") & (F.desc != "tuning curve")].groupby("desc").agg(files=("path", "size"), cells=("cell", "nunique"), rep_min=("rep", "min"), rep_max=("rep", "max")))

Clock check. The trigger times must share the clock of the traces. For each sequence, the
across-trial predictable power (Eq. 52) of the membrane potential in the 1 s after the sequence
onset is compared with the 1 s before it (silence). If the triggers were offset from the traces,
the two would be alike. For MGB units the firing rate in the 0.5 s after onset is compared with
the 0.5 s before.

In [ ]:
ck = pd.DataFrame([dict(group=o["group"], path=o["path"], post=o["clock"]["post"], pre=o["clock"]["pre"])
                   for o in A1 + MG if "clock" in o and np.isfinite(o["clock"]["post"])])
for g, d in ck.groupby("group"):
    print(f"{g:10s} files {len(d):3d}: post-onset median {d.post.median():.3f}, pre-onset median {d.pre.median():.3f}; "
          f"post > pre in {(d.post > d.pre).sum()} of {len(d)}")
print("(A1 in mV^2 of predictable power; MGB in spikes/s)")

Duplicates. Recordings deposited twice would have identical traces. Each file's trace extremes and
median, and its length, are compared across distinct files; files that Summary.mat lists under more
than one experiment group are shown apart.

In [ ]:
multi = F.groupby("path").group.agg(list)
multi = multi[multi.apply(len) > 1]
print("files listed in more than one experiment group of Summary.mat:")
print(multi.to_string() if len(multi) else "none")
cells_multi = F.groupby("cell").group.agg(lambda g: sorted(set(g)))
print("cells listed in more than one group:", cells_multi[cells_multi.apply(len) > 1].to_dict())
U = F.drop_duplicates("path")
key = U.assign(k=U.dur.round(4).astype(str) + "|" + U.vmin.round(4).astype(str) + "|" + U.vmax.round(4).astype(str) + "|" + U.vmedian.round(4).astype(str))
dups = key[key.duplicated("k", keep=False)].sort_values("k")
print("distinct files sharing length, extremes and median:", len(dups))
print(dups[["path", "dur", "vmin", "vmax"]].to_string() if len(dups) else "none")
print("distinct A1 whole-cell cells:", F[F.type == "Whole-cell"].cell.nunique(), "| MGB cells:", F[F.type == "Cell-attached"].cell.nunique())

Time resolution. The analysis runs at 1 kHz (helper docstring). On the probes of one recording the
Kruskal-Wallis significance is recomputed at the paper's 0.1 ms on the 10-kHz median-filtered
trace (no shuffles), and the significant periods are compared.

In [ ]:
def kw_fullres(trace10, starts, length, rep):
    X = np.array([trace10[s:s + length] for s in starts], float)
    M = len(starts) // rep
    N = X.shape[0]
    R = stats.rankdata(X, axis=0)
    rbar = R.reshape(M, rep, -1).mean(axis=1)
    kw = 12 * ((rbar - (N + 1) / 2) ** 2).sum(axis=0) / (M * (N + 1))
    pv = stats.chi2.sf(kw, M - 1)
    return ndimage.binary_opening(pv < cd.ALPHA, structure=np.ones(51, bool))   # >= 5.1 ms, as CxtDep.m


path0 = [o for o in A1 if o["group"] == "natural" and o["desc"] != "tuning curve"][0]["path"]
rec0 = cd.load_recording(path0, keep_raw=True)
x10 = ndimage.median_filter(rec0.raw, size=cd.MEDWIN)
segs = [s for s in cd.sequence_segments(rec0) if s["seq"] is not None]
by = {}
for s in segs:
    by.setdefault(s["seq"], []).append(s)
by = {k: v[: rec0.rep] for k, v in by.items()}
agree = []
for pr in cd.build_probes(rec0):
    rep = pr.rep
    starts = []
    for c in pr.contexts:
        st = []
        for seq, pres in by.items():
            ext = [0] + list(seq)
            for k, f in enumerate(seq):
                if f == pr.probe and ext[k] == c:
                    off = int(sum(rec0.durations[g - 1] for g in seq[:k]))
                    st += [s_["on"] + off for s_ in pres]
        starts += st[:rep]
    L = int(pr.duration * cd.SF)
    full = kw_fullres(x10, starts, L, rep)
    low = cd.kw_significance(pr, nshuffle=0)["sig"]
    full_ds = full[::cd.DEC][: len(low)]
    agree.append(dict(probe=pr.probe, frac_sig_10kHz=full.mean(), frac_sig_1kHz=low.mean(), agreement=(full_ds == low[: len(full_ds)]).mean(),
                      any_10kHz=full.any(), any_1kHz=low.any(), last_10kHz=(np.where(full)[0].max() / cd.SF if full.any() else np.nan),
                      last_1kHz=(np.where(low)[0].max() / cd.FS if low.any() else np.nan)))
del rec0, x10
print(path0)
print(pd.DataFrame(agree).round(4).to_string())

## methods-recording-counts

> Of 194 A1 cells recorded in current clamp, 123 met the analysis criterion (39 tested with natural
> sounds, 14 with TORCs, 6 with DMRs, 39 with modulated harmonic tones, 27 with modulated colored
> noise); 14 of 42 voltage-clamped cells and 14 of 16 MGB cells were analysed.

Rule (criterion; CxtDep.m): a probe counts when it follows at least two distinct conditioning
stimuli, each with at least 4 trials in A1 current clamp and at least 10 in MGB, after cutting all
contexts to the smallest number of trials. Both readings of "silence" are run.

In [ ]:
rows = []
for o in A1 + MG:
    if "counts" in o:
        for inc, lst in o["counts"].items():
            for pr, ctx, ntr, dur in lst:
                rows.append(dict(group=o["group"], cell=o["cell"], path=o["path"], desc=o["desc"], silence=inc, probe=pr,
                                 nctx=len(ctx), rep=min(ntr), dur=dur))
C = pd.DataFrame(rows)


def ens_kind(o):
    if o["group"] == "natural":
        return "natural"
    d = " ".join(st[1] for st in o.get("stim", []))
    for key, name in (("TORC", "TORC"), ("DMR", "DMR"), ("Modulated Harmonic Tone", "MHT"), ("pink noise", "MCN")):
        if key in d:
            return name
    return "naturalistic" if "naturalistic" in o["desc"] else "natural sound variants"


KIND = {o["path"]: ens_kind(o) for o in A1 if "counts" in o}
C["kind"] = C.path.map(KIND).fillna("mgb")
C["ok"] = np.where(C.group == "mgb", C.rep >= 10, C.rep >= 4)
cnt = C[C.ok].groupby(["group", "silence"]).agg(probes=("probe", "size"), cells=("cell", "nunique"))
print(cnt)
allc = {o["cell"] for o in A1 if o["group"] in ("natural", "synthetic", "various")}
for inc in (True, False):
    okc = set(C[(C.ok) & (C.group != "mgb") & (C.silence == inc)].cell)
    print(f"silence counted={inc}: A1 cells meeting the criterion (natural and synthetic ensembles) {len(okc)} of {len(allc)} whole-cell cells in the data set")
print("probes in the data set failing the rule:", int((~C.ok).sum()))
print("A1 cells in the data set:", {g: len({o['cell'] for o in A1 if o['group'] == g}) for g in ('natural', 'synthetic', 'various')},
      "| MGB cells:", len({o['cell'] for o in MG}))
print("voltage-clamp recordings in the data set: 0 (see the data check)")
okk = C[C.ok & C.silence & (C.group != "mgb")]
byk = okk.groupby("kind").agg(probes=("probe", "size"), cells=("cell", "nunique"))
byk["paper_cells"] = byk.index.map({"natural": 39, "TORC": 14, "DMR": 6, "MHT": 39, "MCN": 27}).astype("float")
print("\nA1 cells meeting the rule by ensemble (stimulus descriptions):")
print(byk.to_string())
print("sum of cells over ensembles:", int(byk.cells.sum()), "| over the paper's five ensembles:",
      int(byk.loc[[k for k in ("natural", "TORC", "DMR", "MHT", "MCN") if k in byk.index]].cells.sum()),
      "| distinct cells in those five:", okk[okk.kind.isin(["natural", "TORC", "DMR", "MHT", "MCN"])].cell.nunique())
print("the paper's per-ensemble cell counts (39 + 14 + 6 + 39 + 27) sum to", 39 + 14 + 6 + 39 + 27, "for 123 cells")
# the cells that make the difference between 132 and 123
okg = okk.groupby("cell").agg(groups=("group", lambda g: "+".join(sorted(set(g)))), kinds=("kind", lambda k: "+".join(sorted(set(k)))),
                              probes=("probe", "size"))
print("\nA1 cells meeting the rule, by experiment group of Summary.mat:", okg.groups.value_counts().to_dict())
print("distinct cells in the natural and synthetic groups (the paper's 'Asari and Zador' groups):",
      okk[okk.group.isin(["natural", "synthetic"])].cell.nunique(),
      "| in the 'various acoustic stimuli' group (reference 'Unpublished') only:", int((okg.groups == "various").sum()))
print("cells that meet the rule only through designs outside the paper's five ensembles:")
print(okg[~okg.index.isin(okk[okk.kind.isin(["natural", "TORC", "DMR", "MHT", "MCN"])].cell)].to_string())

**Result: partially reproduced.** The natural sound sample is exact: 305 probes in 39 cells under
both readings of silence. The MGB sample is exact: 93 probes in 14 cells. In A1, 132 of the 145
whole-cell cells in the data set meet the rule, against 123, all of them in the natural and synthetic
groups of Summary.mat. Of the 132, 14 meet it only through designs outside the paper's five
ensembles (9 recordings of one natural sound at several attenuations or band shifts, and 5 of
naturalistic variants); without them 118 distinct cells remain, 5 fewer than 123. Which of these the
paper counted cannot be told from the files. By ensemble, natural sounds (39) and TORCs (14) match,
MCN gives 26 against 27, and DMR (15 against 6) and MHT (26 against 39) depend on how the mixed
designs, a natural probe after DMR or MHT variants, are assigned, which the files do not say. The
two cells listed in both the natural and the synthetic group of Summary.mat agree with the paper's
ensemble counts summing to 125 for 123 cells. The voltage-clamp counts (29 probes in 14 cells)
cannot be tested: the data set holds no voltage-clamp recording.

## a1-mgb-firing-rates

> Firing rates were low in A1 (spontaneous 0.47 +/- 0.61 Hz, evoked 0.57 +/- 0.77 Hz, mean +/- SD, 194
> cells) and high in MGB (spontaneous 0.78 +/- 1.25 Hz, evoked 11.4 +/- 16.9 Hz, 16 cells).

Spontaneous: (a) all silences between stimuli, less 0.5 s after each test pulse, or (b) the last
1 s before each stimulus. Evoked: all stimulus periods (which are the probe periods of the
analysis, every fragment being a probe). A1 spikes, which the paper detected by hand, are peaks at
least 10, 15 or 20 mV above the 10-ms median-filtered trace (an assumption; readings shown). MGB
spikes use the automatic detector of GetSpikes.m. Rates are per cell over all its files, in the
sequence files only, or (MGB) in the tuning-curve files only.

MGB detector check (second cell): GetSpikes.m computes one threshold for the whole trace passed to
it, and ViewResp.m passes the whole recording, so the threshold is set per file. The cell prints,
for each file, the threshold, the robust SD of the filtered trace and the heights of its local
maxima, and runs a second detector at half the automatic threshold as a reading.

In [ ]:
def cell_rates(objs, key=None, files="all"):
    rows = []
    for o in objs:
        if files == "seq" and o["desc"] == "tuning curve":
            continue
        if files == "tuning" and o["desc"] != "tuning curve":
            continue
        r = o["rates"][key] if key is not None else o["rates"]
        rows.append(dict(cell=o["cell"], **r))
    d = pd.DataFrame(rows).groupby("cell").sum()
    return pd.DataFrame(dict(spont=d.n_sp / d.t_sp, pre=d.n_pre / d.t_pre, evoked=d.n_ev / d.t_ev))


out = []
for h in (10, 15, 20):
    for files in ("all", "seq"):
        r = cell_rates(A1, h, files)
        out.append(dict(region="A1", detector=f"{h} mV", files=files, cells=len(r), spont_mean=r.spont.mean(), spont_sd=r.spont.std(),
                        pre_mean=r.pre.mean(), pre_sd=r.pre.std(), evoked_mean=r.evoked.mean(), evoked_sd=r.evoked.std()))
for files in ("all", "seq", "tuning"):
    r = cell_rates(MG, None, files)
    out.append(dict(region="MGB", detector="GetSpikes auto", files=files, cells=len(r), spont_mean=r.spont.mean(), spont_sd=r.spont.std(),
                    pre_mean=r.pre.mean(), pre_sd=r.pre.std(), evoked_mean=r.evoked.mean(), evoked_sd=r.evoked.std()))
    r = cell_rates([dict(o, rates=o["rates5"]) for o in MG], None, files)
    out.append(dict(region="MGB", detector="half threshold", files=files, cells=len(r), spont_mean=r.spont.mean(), spont_sd=r.spont.std(),
                    pre_mean=r.pre.mean(), pre_sd=r.pre.std(), evoked_mean=r.evoked.mean(), evoked_sd=r.evoked.std()))
RATES = pd.DataFrame(out)
print(RATES.round(3).to_string())
print("\npaper: A1 spont 0.47 +/- 0.61, evoked 0.57 +/- 0.77 (194 cells); MGB spont 0.78 +/- 1.25, evoked 11.4 +/- 16.9 Hz (16 cells)")
print("tolerances: A1 0.423-0.517 and 0.513-0.627; MGB 0.702-0.858 and 10.26-12.54 Hz")
mg_seq = cell_rates(MG, None, "seq")
print("\nMGB per cell (sequence files), Hz:\n", mg_seq.round(2).to_string())

In [ ]:
# MGB detector check: GetSpikes.m sets one threshold per file (two thirds of the largest filtered deflection).
# Heights of the local maxima of the filtered trace, in robust SDs, show where the spikes lie relative to it.
rows = []
for o in MG:
    h, ts = o["peak_sd"], o["threshold"] / o["noise_sd"]
    rows.append(dict(file=o["path"].split("/")[-1], desc=o["desc"][:24], minutes=round(o["dur"] / 60, 1), threshold_mV=o["threshold"],
                     noise_sd_mV=o["noise_sd"], threshold_sd=ts, peaks_10_30sd=int(((h >= 10) & (h < 30)).sum()),
                     peaks_30sd_to_half_thr=int(((h >= 30) & (h < ts / 2)).sum()), peaks_half_to_thr=int(((h >= ts / 2) & (h < ts)).sum()),
                     peaks_above_thr=int((h >= ts).sum()), spikes=o["nspk"], spikes_half_thr=o["nspk5"], rate_hz=o["nspk"] / o["dur"]))
thr = pd.DataFrame(rows)
print(thr.round(3).to_string())
print("\nthreshold in robust SDs of the filtered trace: median %.0f, range %.0f to %.0f" % (thr.threshold_sd.median(), thr.threshold_sd.min(), thr.threshold_sd.max()))
print("local maxima over all files: 10-30 SD %d; 30 SD to half the threshold %d; half the threshold to the threshold %d; above it %d"
      % tuple(thr[c].sum() for c in ("peaks_10_30sd", "peaks_30sd_to_half_thr", "peaks_half_to_thr", "peaks_above_thr")))
print("spikes over all files: GetSpikes %d, at half its threshold %d (ratio %.3f)" % (thr.spikes.sum(), thr.spikes_half_thr.sum(), thr.spikes_half_thr.sum() / thr.spikes.sum()))


**Result: partially reproduced, resting on an assumption (the A1 spike detector).** No single
reading puts all four means within tolerance. In A1, with the 15-mV detector on the sequence
recordings, the spontaneous and evoked means are 0.488 and 0.541 Hz (SD 0.608 and 0.677), both
within tolerance; over all recordings the spontaneous mean is 0.363 Hz. In MGB, over all recordings,
the spontaneous rate is 0.618 Hz over all silences or 0.817 Hz in the second before each stimulus,
and the evoked rate is 5.07 Hz (SD 11.46), against 0.78 and 11.4 Hz; the tuning-curve recordings
alone give 11.66 Hz evoked. The automatic threshold lies at 76 to 235 robust SDs of the filtered
trace (median 153); over all files 219802 local maxima lie above it and 59180 between half of it
and it, so in some files it leaves out large deflections. At half the threshold the MGB means are
0.822 Hz spontaneous and 7.15 Hz evoked over all recordings, and 1.03 and 13.37 Hz on the
tuning-curve recordings, so the evoked mean is still outside tolerance. Under the all-recordings
reading the A1 means are below 1 Hz and the MGB evoked mean is above 5 Hz with either MGB detector,
the claim's rule for partial; on the sequence recordings alone the MGB evoked mean is 4.99 Hz with
the automatic detector and 7.07 Hz at half its threshold.

## Natural sound probes in A1

All natural-sound A1 probes, with "silence" counted among the contexts (main reading) and not
counted. The silence probes (probe 0, the first 2.5 s after a sequence) are kept apart for
figs1-silence-probe.

In [ ]:
NP = [p for o in A1 if o["group"] == "natural" for p in o.get("probes", [])]
NPf = {inc: [p for p in NP if p["silence"] == inc and p["probe"] != 0 and p["rep"] >= 4] for inc in (True, False)}
SILP = [p for p in NP if p["silence"] and p["probe"] == 0]
for inc in (True, False):
    d = pd.DataFrame([dict(cell=p["cell"], dur=p["dur"], nctx=len(p["ctx"]), rep=p["rep"]) for p in NPf[inc]])
    print(f"silence counted={inc}: {len(d)} probes in {d.cell.nunique()} cells; contexts per probe {d.nctx.min()}-{d.nctx.max()} "
          f"(median {d.nctx.median():.0f}); trials per context {d.rep.min()}-{d.rep.max()}; probe length {d.dur.min():.2f}-{d.dur.max():.2f} s")
print("silence probes:", len(SILP), "in", len({p['cell'] for p in SILP}), "cells")
TMAX = 6.1
TT = int(TMAX * cd.FS)
t_ax = np.arange(TT) / cd.FS

## fig2-example-reliability

> The example A1 neuron of Figure 2 had high trial-to-trial reliability (correlation across trials
> 0.74 +/- 0.08 for the shown probe and 0.61 +/- 0.07 over the seven natural sound fragments tested),
> and the mean correlation over the population was 0.31 +/- 0.09.

Correlation between the median-filtered traces of the trials of one context over the probe period.
Readings: (a) mean over all pairs of trials, (b) mean correlation of each trial with the mean of
the others; population mean over cells (each cell the mean over its probes and contexts) or over
probes. Silence counted as a context (it is a context of Figure 2B).

In [ ]:
rel = []
for p in NPf[True]:
    for c, X in zip(p["ctx"], p["traces"]):
        if X.shape[0] >= 2:
            a, b = cd.trial_correlations(X.astype(float))
            rel.append(dict(cell=p["cell"], path=p["path"], desc=p["desc"], probe=p["probe"], ctx=c, ntr=X.shape[0], pair=a, loo=b))
REL = pd.DataFrame(rel)
per_probe = REL.groupby(["path", "probe"])[["pair", "loo"]].mean()
per_cell = REL.groupby("cell")[["pair", "loo"]].mean()
for r in ("pair", "loo"):
    print(f"reading {r}: over cells mean {per_cell[r].mean():.3f} +/- {per_cell[r].std():.3f} (n={len(per_cell)}); "
          f"over probes {per_probe[r].mean():.3f} +/- {per_probe[r].std():.3f} (n={len(per_probe)}); "
          f"range over cells {per_cell[r].min():.2f} to {per_cell[r].max():.2f}")
print("paper: 0.31 +/- 0.09 (tolerance 0.22 to 0.40)")
# example cell: seven natural fragments, six repeats
cand = REL[REL.desc.str.contains("7 long")]
cc = cand.groupby("path").agg(cell_mean=("pair", "mean"), cell_mean_loo=("loo", "mean"), max_ctx=("pair", "max"), ntr_max=("ntr", "max"))
cc = cc[cc.ntr_max >= 6]
print("\ncandidate example cells (7 natural fragments, >= 6 trials in a context):", len(cc))
print(cc.round(3).to_string())
m = cc[(cc.cell_mean.between(0.54, 0.68)) | (cc.cell_mean_loo.between(0.54, 0.68))]
print("cells whose mean is within 0.61 +/- 0.07:", len(m))
for path in m.index:
    d = cand[cand.path == path]
    print(path, "probe-context values within 0.74 +/- 0.08:", int(d.pair.between(0.66, 0.82).sum()), "(pairs),", int(d.loo.between(0.66, 0.82).sum()), "(loo)")

**Result: reproduced.** The mean pairwise trial correlation is 0.310 +/- 0.098 over 39 cells (0.319 +/-
0.129 over 305 probes), against 0.31 +/- 0.09; the leave-one-out reading gives 0.436. One recording
with seven fragments and six trials (20051222-ha1-001) has a cell mean of 0.603 and 15 probe-context
values between 0.66 and 0.82, consistent with the example cell (0.61 and 0.74).

## fig2-example-long-context

> In an example A1 neuron, changing the conditioning stimulus changed the response to the probe for
> about 4 s; responses did not differ between 2 and 4 s after probe onset and diverged again after
> about 4 s.

Matching (criterion, as amended after the reader's answer of 2026-11-15): a current-clamp cell
tested with seven natural sound fragments whose mean trial-to-trial reliability is 0.61 (within
0.061), and in it a probe tested after silence and two other fragments, with six trials in at
least one context, whose reliability is 0.74 (within 0.074). The legend gives the cell value as
the spread "across all natural sound stimuli examined in this cell" and the Results "for the seven
natural sound fragments tested in this cell", so the cell value is the mean over its seven
fragments, each the mean over its contexts (main reading); the mean over all probe-context pairs
is shown as a variant. Reliability is the mean pairwise correlation between trials of one context,
as in fig2-example-reliability; the leave-one-out reading of that claim is run as a second reading.
Reproduced when the matched probe has significant periods before 2 s, none between 2 and 3.6 s,
and one after 3.6 s. The earlier strict reading (none between 2 and 4 s, one after 4 s) is shown
for comparison.

In [ ]:
ex = []
for p in NPf[True]:
    s = p["sig"]
    t = np.arange(len(s)) / cd.FS
    ex.append(dict(path=p["path"], probe=p["probe"], desc=p["desc"], ctx=p["ctx"], ntr=p["ntr"], dur=p["dur"],
                   before2=bool(s[t < 2].any()), mid=bool(s[(t >= 2) & (t < 3.6)].any()), after=bool(s[t >= 3.6].any()),
                   mid4=bool(s[(t >= 2) & (t <= 4)].any()), after4=bool(s[t > 4].any()),
                   last=(t[s].max() if s.any() else np.nan)))
EX = pd.DataFrame(ex)
design = EX.desc.str.contains("7 long") & EX.ctx.apply(lambda c: len(c) == 3 and 0 in c) & EX.ntr.apply(lambda n: max(n) >= 6)
print("probes with the example's design (seven fragments, silence and two fragments, >= 6 trials):", int(design.sum()),
      "in", EX[design].path.nunique(), "recordings")
R7 = REL[REL.desc.str.contains("7 long")]
rows = []
for r in ("pair", "loo"):
    for nosil in (False, True):
        d = R7[R7.ctx != 0] if nosil else R7
        pp = d.groupby(["path", "probe"])[r].mean()
        for over in ("probes", "probe-contexts"):
            cm = pp.groupby(level=0).agg(["mean", "std"]) if over == "probes" else d.groupby("path")[r].agg(["mean", "std"])
            cells = cm[(cm["mean"] - 0.61).abs() <= 0.061].index
            m = EX[design & EX.path.isin(cells)].copy()
            m["rel"] = [pp.loc[(a, b)] for a, b in zip(m.path, m.probe)]
            m = m[(m.rel - 0.74).abs() <= 0.074]
            rows.append(dict(reliability=r, silence_ctx=not nosil, cell_mean_over=over, cells_in_range=len(cells),
                             cell_values=", ".join(f"{p.split('/')[1]} {cm.loc[p, 'mean']:.3f} +/- {cm.loc[p, 'std']:.3f}" for p in cells),
                             matches=list(zip(m.path, m.probe)),
                             matched_probes=", ".join(f"{a.split('/')[1]} probe {b} ({v:.3f})" for a, b, v in zip(m.path, m.probe, m.rel)),
                             reproduced=bool(len(m)) and bool((m.before2 & ~m.mid & m.after).all()),
                             strict_2_4=bool(len(m)) and bool((m.before2 & ~m.mid4 & m.after4).all())))
MR = pd.DataFrame(rows)
print(MR.drop(columns="matches").to_string())
print("\nleave-one-out cell means of the seven-fragment recordings (over probes), range:",
      f"{R7.groupby(['path', 'probe']).loo.mean().groupby(level=0).mean().min():.3f} to {R7.groupby(['path', 'probe']).loo.mean().groupby(level=0).mean().max():.3f}")
# the example: the probe matched under the main reading (pairs, silence counted, cell mean over its fragments)
assert len(MR.matches[0]) == 1
ex_path, ex_probe = MR.matches[0][0]
pex = [q for q in NPf[True] if q["path"] == ex_path and q["probe"] == ex_probe][0]
s = pex["sig"]
dd = np.diff(np.r_[0, s.astype(int), 0])
runs = list(zip(np.where(dd == 1)[0] / cd.FS, np.where(dd == -1)[0] / cd.FS))
print(f"\nmatched probe {ex_path} probe {ex_probe}: contexts {pex['ctx']}, trials {pex['ntr']}, length {pex['dur']:.2f} s")
print("  reliability per context (pairs):", REL[(REL.path == ex_path) & (REL.probe == ex_probe)].pair.round(3).tolist())
print("  significant runs (s):", ", ".join(f"{a:.3f}-{b:.3f}" for a, b in runs))
gap = [(b, a2) for (a, b), (a2, b2) in zip(runs[:-1], runs[1:]) if a2 - b > 1]
print("  longest gap between runs: %.3f to %.3f s" % max(gap, key=lambda g: g[1] - g[0]))
print(f"\nall {len(EX)} probes: significant period later than 4 s in", int((EX.last > 4).sum()),
      "; before 2 s, none in 2-3.6 s and one after 3.6 s in", int((EX.before2 & ~EX.mid & EX.after).sum()),
      "; the strict 2-4 s gap pattern in", int((EX.before2 & ~EX.mid4 & EX.after4).sum()))

In [ ]:
# the matched probe at the paper's 0.1-ms resolution (kw_fullres of the time-resolution check)
recx = cd.load_recording(ex_path, keep_raw=True)
x10 = ndimage.median_filter(recx.raw, size=cd.MEDWIN)
byx = {}
for s_ in cd.sequence_segments(recx):
    if s_["seq"] is not None:
        byx.setdefault(s_["seq"], []).append(s_)
byx = {k: v[: recx.rep] for k, v in byx.items()}
starts = []
for c in pex["ctx"]:
    st = []
    for seq, pres in byx.items():
        ext = [0] + list(seq)
        for k, f in enumerate(seq):
            if f == ex_probe and ext[k] == c:
                st += [s_["on"] + int(sum(recx.durations[g - 1] for g in seq[:k])) for s_ in pres]
    starts += st[: pex["rep"]]
full = kw_fullres(x10, starts, int(pex["dur"] * cd.SF), pex["rep"])
tf = np.arange(len(full)) / cd.SF
print("0.1-ms resolution: significant before 2 s", bool(full[tf < 2].any()), "; between 2 and 3.6 s", bool(full[(tf >= 2) & (tf < 3.6)].any()),
      "; after 3.6 s", bool(full[tf >= 3.6].any()), "; last before 3.6 s %.4f s; first after 2 s %.4f s" % (tf[full & (tf < 3.6)].max(), tf[full & (tf >= 2)].min()))
del recx, x10
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for c, X in zip(pex["ctx"], pex["traces"]):
    ax[0].plot(np.arange(X.shape[1]) / cd.FS, X.mean(0), lw=1, label=f"context {c}" + (" (silence)" if c == 0 else ""))
ax[0].legend(fontsize=7); ax[0].set_ylabel("mean Vm (mV)")
ax[1].semilogy(np.arange(len(pex["pval"])) / cd.FS, pex["pval"], lw=0.5)
ax[1].fill_between(np.arange(len(pex["sig"])) / cd.FS, 1e-6, 1, where=pex["sig"], color="0.8")
for x in (2, 3.6, 4):
    ax[1].axvline(x, color="r" if x == 3.6 else "0.5", lw=0.8, ls="--")
ax[1].set_ylabel("P (Kruskal-Wallis)"); ax[1].set_xlabel("time from probe onset (s)")
print("plotted:", ex_path, "probe", ex_probe, "; Vm range", round(float(min(X.min() for X in pex["traces"])), 2), "to",
      round(float(max(X.max() for X in pex["traces"])), 2), "mV; P from", float(pex["pval"].min()), "to 1, axis floor 1e-6")
plt.show()

**Result: reproduced, resting on an assumption (reliability as the mean correlation over pairs of
trials).** With pairwise reliability, one cell of the 10 with seven fragments has a mean within 0.61
+/- 0.061 (20051222-ha1-001, 0.611 +/- 0.081 over its seven fragments, 0.603 over its probe-context
pairs), and one of its probes matches (probe 4, contexts silence, 2 and 3, six trials each,
reliability 0.737). Its significant periods run up to 1.601 s, none from 1.601 to 3.971 s, and
again from 3.971 to 4.597 s, so it is significant before 2 s, not between 2 and 3.6 s, and again
after 3.6 s. The same probe is matched with the silence context left out of the reliability (0.716),
and the 0.1-ms resolution gives the same gap (1.6004 to 3.9696 s). Under the strict reading of the
earlier criterion, none between 2 and 4 s, this probe fails, as its renewed run starts 29 ms before
4 s. With leave-one-out reliability no cell matches (cell means 0.296 to 0.732, the example cell at
0.725), so the claim would not be testable; that reading gives a population mean of 0.436 against
the paper's 0.31 (fig2-example-reliability), which is why the pairwise reading is adopted. As the
weaker check, 18 of 305 probes have a significant period later than 4 s.


## fig3a-proportion-significant

> Of 305 natural sound probes tested in 39 A1 cells with typically five to eight natural sound
> contexts, 204 (66.9%) showed significant context dependence, and about a quarter (23.7%) of the
> significant events occurred more than 1 s after probe onset.

A probe is significant when it has at least one period with P < 0.01 for at least 5 ms. Late
fraction: one minus the cumulative probability at 1 s, the cumulative probability being the
running integral of the proportion curve less the noise floor, normalized at its peak (Figure 3A
legend). Readings: silence counted or not; proportion over all probes (a probe that has ended
counts as not significant) or over the probes still running.

In [ ]:
def pop_curves(P, denom):
    Sg = cd.stack_curves([p["sig"].astype(float) for p in P], TT)
    Nf = cd.stack_curves([p["noise"] for p in P], TT)
    if denom == "all":
        return np.nan_to_num(Sg).sum(0) / len(P), np.nan_to_num(Nf).sum(0) / len(P), np.ones(TT) * len(P)
    n = (~np.isnan(Sg)).sum(0)
    with np.errstate(invalid="ignore"):
        return np.nansum(Sg, 0) / n, np.nansum(Nf, 0) / n, n


fig3a = []
CURVES = {}
for inc in (True, False):
    P = NPf[inc]
    nsig = sum(bool(p["sig"].any()) for p in P)
    for denom in ("all", "running"):
        prop, nf, n = pop_curves(P, denom)
        ok = n >= 10
        corr = np.where(ok, prop - nf, 0.0)
        cum = np.cumsum(corr) / cd.FS
        ipk = int(np.argmax(cum))
        cumn = cum / cum[ipk]
        CURVES[(inc, denom)] = (prop, nf, cumn, ok)
        fig3a.append(dict(silence=inc, denom=denom, probes=len(P), significant=nsig, fraction=nsig / len(P),
                          late_fraction=1 - cumn[int(1.0 * cd.FS)], cum_peak_s=ipk / cd.FS, last_t_with_10_probes=t_ax[ok].max()))
FIG3A = pd.DataFrame(fig3a)
print(FIG3A.round(4).to_string())
print("paper: 204 of 305 (66.9%), late fraction 0.237, peak of the cumulative curve 4.72 s")
print("tolerances: 183.6-224.4 probes, 0.602-0.736, 0.213-0.261")
# significance by cell
bycell = pd.DataFrame([dict(cell=p["cell"], sig=bool(p["sig"].any())) for p in NPf[True]]).groupby("cell").sig.agg(["sum", "size"])
print(f"\ncells with at least one significant probe: {(bycell['sum'] > 0).sum()} of {len(bycell)}; per-cell fraction median {(bycell['sum'] / bycell['size']).median():.2f}")

In [ ]:
prop, nf, cumn, ok = CURVES[(True, "all")]
fig, ax = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax[0].plot(t_ax, prop, "k", lw=1, label="proportion significant"); ax[0].plot(t_ax, nf, "0.6", lw=1, label="noise floor")
ax[0].legend(fontsize=8); ax[0].set_ylabel("probability")
ax[1].plot(t_ax, cumn, "k"); ax[1].set_ylabel("cumulative (normalized)"); ax[1].set_xlabel("time from probe onset (s)")
print(f"plotted ranges: proportion {prop.min():.3f}-{prop.max():.3f}, noise {nf.min():.4f}-{nf.max():.4f}, cumulative {cumn.min():.3f}-{cumn.max():.3f}")
plt.show()

**Result: reproduced.** With silence counted as a context, 187 of 305 probes (61.3%) have a
significant period, within the tolerance of 204 (66.9%), and the late fraction is 0.251 with the
proportion taken over all probes (peak of the cumulative curve at 4.78 s, paper 4.72 s); over the
running probes it is 0.267, outside. Without silence 169 probes (55.4%) are significant and the late
fraction is 0.177, both outside. 36 of 39 cells have at least one significant probe.

## fig3a-significance-decay

> The proportion of probes with significant context dependence decays after probe onset as the sum
> of two exponentials (sizes 0.17 and 0.09, decay constants 0.20 and 0.90 s), above a noise floor
> of about 2.8 x 10^-3.

Eq. 44 with the constant fixed at the mean noise floor over the fit range, least squares from a
grid of start values (standard choice), two exponentials kept when each size is at least a tenth
of the total. Readings: fit range 0 to 4.72 s or 0 to 6 s; silence counted or not; proportion over
all probes or over running ones.

In [ ]:
fits = []
for (inc, denom), (prop, nf, cumn, ok) in CURVES.items():
    for tmax in (4.72, 6.0):
        m = (t_ax <= tmax) & ok
        kappa = float(np.mean(nf[m]))
        f, alt = cd.fit_with_rule(t_ax[m], prop[m], kappa=kappa)
        two = f if f["n_exp"] == 2 else alt
        fits.append(dict(silence=inc, denom=denom, t_max=tmax, n_exp=f["n_exp"], kappa=kappa,
                         a1=two["alpha"][0], tau1=two["tau"][0], a2=two["alpha"][1], tau2=two["tau"][1],
                         one_a=(f if f["n_exp"] == 1 else alt)["alpha"][0] if f["n_exp"] == 1 else np.nan,
                         one_tau=(f if f["n_exp"] == 1 else alt)["tau"][0] if f["n_exp"] == 1 else np.nan))
FIT3A = pd.DataFrame(fits)
print(FIT3A.round(4).to_string())
print("paper: a1 0.17, tau1 0.20 s, a2 0.09, tau2 0.90 s, kappa 2.8e-3")
print("tolerances: a1 0.153-0.187, tau1 0.18-0.22, a2 0.081-0.099, tau2 0.81-0.99, kappa 0.00252-0.00308")

**Result: partially reproduced.** Two exponentials are selected under every reading with silence
counted. Over all probes from 0 to 4.72 s the sizes are 0.151 and 0.079 and the constants 0.253 and
0.974 s, with a constant of 0.0026: the slow constant and the noise floor are within tolerance, the
fast size, the fast constant and the slow size are not. Over the running probes the fast size is
within tolerance (0.162) and the slow constant is 1.09 s. Without silence one exponential is
selected (size 0.166, constant 0.316 s).

## fig3b-fractional-power-decay

> The context-dependent fraction of the stimulus-related response power, normalized by the
> smoothed stimulus-related power, decays from about 0.49 with a time constant of about 1.04 s.

Q(t) of Eq. 20: Eq. 19 over Eq. 17, both averaged over probes at each time, the denominator
smoothed by a running mean over [0, 2t]; one exponential with constant 0. Readings (claims file):
silence counted or not; the probes that enter the average at time t, those still running at t or
all of them with ended probes as zero (which changes the ratio through the smoothing once t passes
the end of the shortest probe); fit range 0 to 4 s or 0 to 6 s. A fit to the shortest probe
(1.98 s), a range the claims file does not list, is shown for comparison.

In [ ]:
POW = {}
for inc in (True, False):
    P = NPf[inc]
    for denom in ("running", "all"):
        st = {k: cd.stack_curves([p[k] for p in P], TT) for k in ("pred", "inv", "dep")}
        if denom == "running":
            av = {k: np.nanmean(v, 0) for k, v in st.items()}
        else:   # every probe counts at every t, an ended probe as zero
            av = {k: np.nan_to_num(v).sum(0) / len(P) for k, v in st.items()}
        POW[(inc, denom)] = dict(**av, ps=cd.running_mean_0_2t(av["pred"]), shortest=min(p["dur"] for p in P))
rows = []
for (inc, denom), d in POW.items():
    Q = d["dep"] / d["ps"]
    for tmax in (4.0, 6.0, d["shortest"]):
        m = (t_ax <= tmax) & np.isfinite(Q)
        f = cd.fit_exp(t_ax[m], Q[m], 1, kappa=0.0)
        rows.append(dict(silence=inc, probes_at_t=denom, t_max=round(tmax, 2), alpha1=f["alpha"][0], tau1=f["tau"][0],
                         Q_first_50ms=float(np.mean(Q[:50])), Q_at_1s=float(Q[1000]), Q_at_3s=float(Q[3000])))
FIT3B = pd.DataFrame(rows)
print(FIT3B.round(3).to_string())
print("paper: alpha1 0.49 (0.441-0.539), tau1 1.04 s (0.936-1.144)")
print("readings with both values within tolerance:", int(((FIT3B.alpha1 - 0.49).abs() <= 0.049).mul((FIT3B.tau1 - 1.04).abs() <= 0.104).sum()))


In [ ]:
d = POW[(True, "running")]
Q = d["dep"] / d["ps"]
fig, ax = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax[0].plot(t_ax, d["pred"], "k", lw=0.5, label="stimulus-related (Eq. 17)"); ax[0].plot(t_ax, d["ps"], "0.5", label="running mean [0, 2t]")
ax[0].plot(t_ax, d["dep"], "k", lw=2, label="context-dependent (Eq. 19)"); ax[0].legend(fontsize=7); ax[0].set_ylabel("mV^2")
ax[1].plot(t_ax, Q, "k"); ax[1].set_ylim(-0.2, 1.0); ax[1].set_ylabel("Q(t)"); ax[1].set_xlabel("time from probe onset (s)")
print(f"ranges: Eq.17 {np.nanmin(d['pred']):.2f}-{np.nanmax(d['pred']):.2f} mV^2, Eq.19 {np.nanmin(d['dep']):.2f}-{np.nanmax(d['dep']):.2f}; "
      f"Q {np.nanmin(Q):.3f}-{np.nanmax(Q):.3f}, points outside -0.2..1: {int(((Q < -0.2) | (Q > 1)).sum())}")
plt.show()

**Result: not reproduced.** Q(t) falls from 0.76 in the first 50 ms to 0.19 at 1 s. Over the
listed readings (silence counted or not, running or all probes, 0 to 4 or 6 s) the fitted size is
0.448 to 0.470, within tolerance, and the time constant is 1.27 to 1.78 s, outside 0.936 to 1.144 s:
1.73 to 1.78 s with silence counted and 1.27 to 1.32 s without, and the average over all probes
changes it by at most 0.04 s. No reading puts both values within tolerance, and the criterion has
no rule for partial. Fits to the shortest probe (1.98 s), not a listed range, give 0.73 to 1.16 s
with sizes of 0.549 to 0.580.

## fig4-predictability-upper-bound

> Without knowledge of the context, no model can capture more than about half (0.51) of the
> predictable response power with a window of about 100 ms; the upper bound approaches one only
> with a window of seconds (time constant 1.04 s).

Eq. 43, Eq. 18 over Eq. 17 averaged over probes, with the denominator smoothed over [0, 2t] (main
reading) or not; one exponential with constant 1 (1 + alpha1 exp(-t/tau1)). The readings of
fig3b-fractional-power-decay (silence, probes averaged at t, fit range) are run. As a comparison
outside the criterion, 1 - Q(t) is fitted the same way, since the Figure 4 fit (-0.49, 1.04 s)
mirrors that of Figure 3B.

In [ ]:
rows = []
for (inc, denom), d in POW.items():
    for smooth in (True, False):
        U = d["inv"] / (d["ps"] if smooth else d["pred"])
        for tmax in (4.0, 6.0, d["shortest"]):
            m = (t_ax <= tmax) & np.isfinite(U)
            f = cd.fit_exp(t_ax[m], U[m], 1, kappa=1.0)
            rows.append(dict(silence=inc, probes_at_t=denom, smoothed=smooth, t_max=round(tmax, 2), bound_at_0=1 + f["alpha"][0], tau1=f["tau"][0],
                             U_mean_0_100ms=float(np.mean(U[:100])), U_at_1s=float(U[1000]), U_at_3s=float(U[3000])))
FIT4 = pd.DataFrame(rows)
print(FIT4.round(3).to_string())
print("paper: 1 + alpha1 = 0.51 (0.459-0.561), tau1 1.04 s (0.936-1.144)")
ok4 = ((FIT4.bound_at_0 - 0.51).abs() <= 0.051) & ((FIT4.tau1 - 1.04).abs() <= 0.104)
print("readings with both values within tolerance:", int(ok4.sum()), "of", len(FIT4), "| tau1 within tolerance:", int(((FIT4.tau1 - 1.04).abs() <= 0.104).sum()))
# not a reading of the criterion: the Figure 4 fit (-0.49, 1.04 s) mirrors Figure 3B, as if 1 - Q(t) had been plotted
rows = []
for (inc, denom), d in POW.items():
    V = 1 - d["dep"] / d["ps"]
    m = (t_ax <= 4.0) & np.isfinite(V)
    f = cd.fit_exp(t_ax[m], V[m], 1, kappa=1.0)
    rows.append(dict(silence=inc, probes_at_t=denom, bound_at_0=1 + f["alpha"][0], tau1=f["tau"][0]))
print("\n1 - Q(t), fitted from 0 to 4 s with constant 1 (comparison only):")
print(pd.DataFrame(rows).round(3).to_string())


**Result: not reproduced.** The bound near probe onset is within tolerance in several readings
(1 + alpha1 from 0.459 to 0.534, for example 0.498 with the unsmoothed denominator and silence
counted), and the mean of Eq. 43 over the first 100 ms is 0.492 (smoothed, silence counted). The time
constant is 1.45 to 2.81 s in every fit to 4 or 6 s that converged, none within 0.936 to 1.144 s;
the two fits over all probes to 6 s with the smoothed denominator stop at the 100-s bound. The main
reading (smoothed, silence counted, running probes, 0 to 4 s) gives 0.430 and 2.32 s. No reading
puts both values within tolerance, and the criterion has no rule for partial. The closest constant,
0.928 s, comes from a fit to 1.98 s, a range the claims file does not list, where the bound is
0.439. 1 - Q(t) gives 0.53 to 0.55 at onset and 1.27 to 1.76 s.

## timescale-longer-than-membrane

> The trial-to-trial noise autocorrelation and the autocorrelation of spontaneous activity decay
> within about 100 ms, much faster than the second-long timescale of context dependence in A1.

Noise: residual of each trial from its context mean (Eq. 14) over the probe period, autocorrelation
normalized by its variance (Eq. 45), averaged over trials and probes. Spontaneous: the silences
between sequences (whole, or without their first 1 s), up to the first test pulse in each. Decay:
lag at which the mean autocorrelation first falls below 1/e (main) or 0.1 (variant).

In [ ]:
lags = np.arange(int(0.5 * cd.FS) + 1) / cd.FS
ac_noise = np.mean([cd.noise_autocorr(dict2) for dict2 in [cd.Probe(cell=p["cell"], path=p["path"], probe=p["probe"], contexts=list(p["ctx"]), traces=p["traces"], n_trials=list(p["ntr"]), duration=p["dur"]) for p in NPf[True]]], axis=0)
ac_sp = {k: np.mean([o["spont_ac"][k] for o in A1 if o.get("spont_ac") and o["spont_ac"][k] is not None], axis=0) for k in (0.0, 1.0)}
nsp = {k: sum(1 for o in A1 if o.get("spont_ac") and o["spont_ac"][k] is not None) for k in (0.0, 1.0)}
rows = [dict(series="trial noise (305 probes)", lag_1e=cd.first_cross(lags, ac_noise, 1 / np.e), lag_01=cd.first_cross(lags, ac_noise, 0.1))]
for k in (0.0, 1.0):
    rows.append(dict(series=f"spontaneous, skip {k:.0f} s ({nsp[k]} files)", lag_1e=cd.first_cross(lags, ac_sp[k], 1 / np.e), lag_01=cd.first_cross(lags, ac_sp[k], 0.1)))
TS = pd.DataFrame(rows)
print(TS.to_string())
print("criterion: both 1/e lags < 0.150 s, and below a third of 1.04 s (0.347 s)")
print("autocorrelation at 100 ms: noise %.3f, spontaneous %.3f" % (ac_noise[100], ac_sp[0.0][100]))

**Result: reproduced.** The noise autocorrelation falls below 1/e at 72 ms (below 0.1 at 115 ms)
and that of the spontaneous activity at 78 ms (76 ms without the first 1 s of each silence), all
under 150 ms and under a third of 1.04 s.

## fig9-mgb-fast-decay

> In the auditory thalamus (MGB), context dependence of spike responses decayed within a short time
> (time constant about 80 ms; 93 probes in 14 cells), so the long-lasting component in A1 arises
> mainly in cortex.

PSTH in 100-ms bins from probe onset for each context, SD over contexts (n - 1), averaged over
probes (CxtDep.m, psth); one exponential with a free constant. Readings: bin time at its start or
centre; average over the probes still running or over all (ended probes as zero); silence counted
or not. Spikes from the automatic detector of GetSpikes.m, and at half its threshold as a
second reading (a1-mgb-firing-rates).

In [ ]:
MP = {inc: [p for o in MG for p in o.get("probes", []) if p["silence"] == inc and p["rep"] >= 10] for inc in (True, False)}
for inc in (True, False):
    print(f"silence counted={inc}: {len(MP[inc])} probes in {len({p['cell'] for p in MP[inc]})} cells")
rows = []
SDC = {}
for det, key in (("GetSpikes auto", "sd"), ("half threshold", "sd5")):
    for inc in (True, False):
        Sd = cd.stack_curves([p[key] for p in MP[inc]], 61)
        for denom in ("running", "all"):
            y = np.nanmean(Sd, 0) if denom == "running" else np.nan_to_num(Sd).sum(0) / len(MP[inc])
            SDC[(det, inc, denom)] = y
            for pos in ("start", "centre"):
                tb = np.arange(61) * 0.1 + (0.05 if pos == "centre" else 0.0)
                f = cd.fit_exp(tb, y, 1, kappa=None)
                late = float(np.nanmean(y[tb > 0.5]))
                rows.append(dict(detector=det, silence=inc, denom=denom, bin_time=pos, alpha1=f["alpha"][0], tau1=f["tau"][0], kappa=f["kappa"],
                                 first_bin=y[0], second_bin=y[1], mean_after_0_5s=late, plateau_ratio=late / f["kappa"], late_over_paper_kappa=late / 2.0))
FIT9 = pd.DataFrame(rows)
print(FIT9.round(3).to_string())
print("paper: alpha1 15.0 spike/s (13.5-16.5), tau1 0.080 s (0.072-0.088, or below 0.1 s), kappa 2.0 (1.8-2.2)")
print("plateau_ratio compares the late mean with the fitted kappa, which the fit pulls towards it; late_over_paper_kappa compares it with 2.0")
y = SDC[("GetSpikes auto", True, "running")]
print("mean SD by bin (GetSpikes, silence counted, running probes):", np.round(y[:12], 2))


**Result: reproduced.** The mean SD of the PSTHs over 93 probes in 14 cells falls from 6.45 spikes/s
in the first bin to 2.04 in the second. With silence counted and the running probes, tau1 is 0.058
s and the mean after 0.5 s is 0.99 times the fitted constant; over all probes with bin starts tau1 is
0.080 s. At half the detection threshold tau1 is 0.064 and 0.081 s. The sizes are smaller than the
paper's: alpha1 5.2 to 12.4 spikes/s against 15.0 (7.0 to 15.4 at half the threshold), and kappa 0.81
to 1.21 against 2.0; the mean after 0.5 s is 0.39 to 0.60 of the paper's kappa. The plateau test
compares the late mean with the fitted kappa, which the fit draws towards it. Without silence the
first bin is 2.50, and the fit over all probes does not converge to a decay (tau1 at its bound of
100 s).

## fig9-example-first-bin

> In an example MGB unit, the response to a probe differed between contexts only in the first
> 100-ms bin of the PSTH.

Among MGB probes with the design of Figure 9A (the Figure 2 sequence design, seven long natural
fragments, three contexts of which one is silence, at least 10 trials), reproduced when at least one
has its SD over contexts in the first bin at least twice that of every later bin. The fraction of
all probes with that pattern is reported.

In [ ]:
rows = []
for p in MP[True]:
    sd = p["sd"]
    rows.append(dict(path=p["path"], probe=p["probe"], desc=p["desc"], ctx=p["ctx"], rep=p["rep"], first=sd[0],
                     max_later=sd[1:].max(), ratio=sd[0] / sd[1:].max() if sd[1:].max() > 0 else np.inf))
E9 = pd.DataFrame(rows)
E9["pattern"] = E9.ratio >= 2
des = E9[E9.desc.str.contains("7 long") & E9.ctx.apply(lambda c: len(c) == 3 and 0 in c)]
print("probes with the Figure 9A design:", len(des), "; with the first-bin pattern:", int(des.pattern.sum()))
print(des[["path", "probe", "ctx", "rep", "first", "max_later", "ratio"]].round(2).to_string())
print(f"\nall {len(E9)} probes: pattern in {int(E9.pattern.sum())} ({E9.pattern.mean():.3f}); with factor 1.5: {(E9.ratio >= 1.5).sum()}, factor 3: {(E9.ratio >= 3).sum()}")

**Result: reproduced.** Of 12 probes with the Figure 9A design, 4 (all in 20071010-ha2-003) have a
first-bin SD at least twice that of every later bin (ratios 2.05 to 2.81). Over all 93 probes the
pattern holds in 35 (0.376).

## fig7-example-frequency-context

> In an example A1 neuron, a natural sound probe preceded by three DMR contexts that differed only
> in frequency band gave significant context dependence for about 4 s.

Search (criterion): a recording with DMR contexts in 0.625-2.5, 2.5-10 and 10-40 kHz and a natural
sound probe. Every synthetic recording is listed by its stimulus types; for each DMR or TORC
stimulus the band edges come from its `ff` parameter and, for TORC sequences, from the `ff` of each
TORC (`torc_parameters`). The second cell runs the significance measure on the nearest design.

In [ ]:
# every DMR and TORC stimulus of every synthetic recording, with all its band edges (Hz), and the probes that follow it
target = [(625.0, 2500.0), (2500.0, 10000.0), (10000.0, 40000.0)]
rows = []
for o in A1:
    if o["group"] != "synthetic" or o["desc"] == "tuning curve":
        continue
    st = o["stim"]
    syn = [i for i, s in enumerate(st) if s[3] in ("DMR", "TORC")]
    if not syn:
        continue
    bands = sorted({b for i in syn for b in st[i][4]})
    nat_probe = any(st[i][3] == "natural" for i in range(len(st)))
    rows.append(dict(path=o["path"].split("/")[-1], design=o["desc"][:34], kinds=", ".join(sorted({st[i][1].split(":")[0][:28] for i in syn})),
                     natural_probe=nat_probe, n_bands=len(bands), bands=", ".join("-".join(f"{v / 1000:g}" for v in b) for b in bands),
                     fig7_bands=sum(b in bands for b in target)))
B = pd.DataFrame(rows)
print("synthetic recordings with DMR or TORC stimuli:", len(B), "of", sum(1 for o in A1 if o["group"] == "synthetic" and o["desc"] != "tuning curve"))
print(B.groupby(["design", "kinds", "natural_probe", "bands", "fig7_bands"]).size().rename("recordings").reset_index().to_string())
print("\nrecordings with all three Figure 7 bands (0.625-2.5, 2.5-10, 10-40 kHz) as separate stimuli:", int((B.fig7_bands == 3).sum()),
      "| with a natural probe after DMR frequency variants:", int((B.natural_probe & B.kinds.str.contains("DMR; frequency")).sum()))
other = [o for o in A1 if o["group"] == "synthetic" and o["desc"] != "tuning curve" and not any(s[3] in ("DMR", "TORC") for s in o["stim"])]
print("the other", len(other), "synthetic recordings hold no DMR or TORC stimulus; their stimulus types:",
      dict(collections.Counter(s[3] for o in other for s in o["stim"])))


In [ ]:
# the nearest design (natural probe after DMR frequency variants in other bands): significance with the forward variants only
rows = []
for o in A1:
    for t in o.get("fig8", {}).get("tests", []):
        if t["label"].startswith("DMR") and t["prop"] == "frequency" and t["kind"] == "natural":
            s = t["sig"]
            rows.append(dict(path=t["path"].split("/")[-1], set=t["label"], contexts=len(t["ctx"]), trials=t["rep"], length_s=round(t["dur"], 2),
                             significant=bool(s.any()), last_significant_s=(np.where(s)[0].max() / cd.FS if s.any() else np.nan)))
print("natural probes after DMR frequency variants (0.25-1 to 8-32 kHz; not the Figure 7 bands, so this does not decide the claim):")
print(pd.DataFrame(rows).round(3).to_string())


**Result: not testable (blocked, data missing).** Of the 105 synthetic recordings, 35 hold DMR or TORC
stimuli and the other 70 hold none (MHT, pink noise, natural and naturalistic sounds). None has
DMR contexts in the three Figure 7 bands. The stimuli with edges 0.625, 2.5, 10 and 40 kHz are
single two-band DMRs and TORCs (0.625-2.5 with 10-40 kHz) in designs without a natural probe, and
one recording has a natural probe after DMR amplitude variants in 2.5-10 kHz only. The five
recordings of a natural probe after DMR frequency variants use two-octave bands from 0.25-1 to 8-32
kHz. In these five, with the forward variants as contexts, the probe is significant in 4, last at
1.25 to 3.67 s; this nearest design does not decide the claim.

## Synthetic conditioning stimuli: property sets (Figure 8)

The files name the sets of conditioning stimuli in which one property varies only in the notes of
the six recordings of a natural probe after MHT variants. For the others the sets are rebuilt from
the stimulus descriptions, names and parameters (`ctxdep.property_sets`, whose docstring gives each
rule). This is an assumption the five Figure 8 claims rest on. For each probe and set, the
significance and fractional power measures use only the contexts of that set (claims file), each
with at least 4 trials. Readings: main (one test per probe and property, the five probe types of
Table 1: the forward variants where forward and time-reversed ones exist, the set through the
first stimulus in the 4 by 4 AM by FM designs, and for higher-order the natural sound and pink
noise set that holds the probe); all sets (every set a test); all probe types (main, plus the
probes that are naturalistic variants).

The cell below prints the sets of every design and compares the tests with Table 1.

In [ ]:
T8 = [t for o in A1 if o["group"] == "synthetic" for t in o.get("fig8", {}).get("tests", [])]
PROPS = ["amplitude", "frequency", "AM", "FM", "higher-order"]


def sel8(reading):
    if reading == "main":
        return [t for t in T8 if t["main"] and t["kind"] in cd.TABLE1_KINDS and (t["prop"] != "higher-order" or t["own"])]
    if reading == "all sets":
        return [t for t in T8 if t["kind"] in cd.TABLE1_KINDS]
    return [t for t in T8 if t["main"] and (t["prop"] != "higher-order" or t["own"])]


READ8 = ["main", "all sets", "all probe types"]
# the sets of every design
srows = []
for o in A1:
    if o["group"] == "synthetic" and "fig8" in o:
        st = o["stim"]
        sets = "; ".join(f"{p}: {lab} {list(m)}{'' if mn else ' (variant)'}" for p, lab, m, mn in o["fig8"]["sets"]) or "none"
        srows.append(dict(design=o["desc"][:40], stimuli=len(st), types="+".join(sorted({s[3] for s in st})), sets=sets))
SR = pd.DataFrame(srows).groupby(["design", "stimuli", "types", "sets"]).size().rename("recordings").reset_index()
for r in SR.itertuples():
    print(f"{r.recordings:3d} recordings | {r.design} | {r.stimuli} stimuli ({r.types})\n      sets: {r.sets}")
print("recordings with no property set:", int(SR[SR.sets == "none"].recordings.sum()))
TAB1 = {("amplitude", "natural"): (23, 23), ("frequency", "natural"): (25, 25), ("AM", "natural"): (25, 25), ("FM", "natural"): (25, 25),
        ("higher-order", "natural"): (63, 27), ("amplitude", "TORC"): (39, 9), ("frequency", "TORC"): (40, 9), ("amplitude", "DMR"): (8, 2),
        ("frequency", "DMR"): (20, 5), ("amplitude", "MHT"): (25, 20), ("frequency", "MHT"): (25, 21), ("AM", "MHT"): (71, 27),
        ("FM", "MHT"): (57, 25), ("higher-order", "MCN"): (74, 27)}
for rd in READ8:
    d = pd.DataFrame([dict(prop=t["prop"], kind=t["kind"], cell=t["cell"], sig=bool(t["sig"].any())) for t in sel8(rd)])
    g = d.groupby(["prop", "kind"]).agg(tests=("sig", "size"), cells=("cell", "nunique"), significant=("sig", "sum")).reset_index()
    g["table1_probes"] = [TAB1.get((p, k), (np.nan,))[0] for p, k in zip(g.prop, g.kind)]
    g["table1_cells"] = [TAB1.get((p, k), (np.nan, np.nan))[1] for p, k in zip(g.prop, g.kind)]
    print(f"\nreading: {rd}")
    print(g.to_string())


## fig8a-proportion-by-property

> When only one property of synthetic conditioning stimuli varied, context dependence was found in
> 77 of 95 probes (81.1%, 31 cells) for amplitude, 73 of 110 (35 cells) for frequency, 62 of 96
> (64.6%, 27 cells) for AM, 47 of 82 (57.3%, 25 cells) for FM and 59 of 137 (43.1%, 27 cells) for
> higher-order properties.

Reproduced when all five counts are within tolerance; partially reproduced when the proportions
keep the order amplitude > frequency and AM > FM > higher-order.

In [ ]:
paper8 = {"amplitude": (77, 95), "frequency": (73, 110), "AM": (62, 96), "FM": (47, 82), "higher-order": (59, 137)}
rows = []
for rd in READ8:
    d = pd.DataFrame([dict(prop=t["prop"], cell=t["cell"], sig=bool(t["sig"].any())) for t in sel8(rd)])
    g = d.groupby("prop").agg(tests=("sig", "size"), cells=("cell", "nunique"), significant=("sig", "sum"))
    g["fraction"] = g.significant / g.tests
    fr = g.fraction
    for p in PROPS:
        rows.append(dict(reading=rd, prop=p, tests=int(g.tests.get(p, 0)), cells=int(g.cells.get(p, 0)), significant=int(g.significant.get(p, 0)),
                         fraction=float(fr.get(p, np.nan)), paper=f"{paper8[p][0]}/{paper8[p][1]} ({paper8[p][0] / paper8[p][1]:.3f})",
                         count_in_tolerance=abs(g.significant.get(p, 0) - paper8[p][0]) <= 0.1 * paper8[p][0]))
    order = (fr.get("amplitude", np.nan) > fr.get("frequency", np.nan)) and (fr.get("AM", np.nan) > fr.get("FM", np.nan) > fr.get("higher-order", np.nan))
    rows.append(dict(reading=rd, prop="order amplitude > frequency, AM > FM > higher-order", fraction=float(order)))
F8A = pd.DataFrame(rows)
print(F8A.round(3).to_string())


**Result: partially reproduced, resting on an assumption (the property sets).** Main reading:
amplitude 43 of 70 tests significant (0.614, 30 cells), frequency 52 of 85 (0.612, 34), AM 19 of 39
(0.487, 26), FM 12 of 25 (0.480, 24) and higher-order 41 of 131 (0.313, 26). Every count is outside
its tolerance, as the data set holds fewer tests: the MHT probes of Table 1 (25 amplitude, 25
frequency, 71 AM and 57 FM) are represented by 14 AM tests in 2 cells. The proportions keep the
order amplitude > frequency and AM > FM > higher-order, by 0.002 and 0.007 for the first two
comparisons. With every set as a test the order fails (frequency 0.632 above amplitude 0.608); with
the naturalistic probes added it holds (0.600 > 0.582; 0.487 > 0.480 > 0.313). Where the data hold
the cells, the main reading gives the rows of Table 1 closely: natural AM and FM 25 tests in 24
cells (25 in 25), DMR 8 in 2 and 20 in 5 (the same), TORC 42 and 44 in 9 (39 and 40 in 9), MCN 73
in 25 (74 in 27) and natural higher-order 58 in 25 (63 in 27).

## fig8a-significance-fits

> The proportion of probes with significant context dependence decays fastest and is smallest for
> higher-order changes, with fitted (size, time constant) of (0.23, 0.25 s) for amplitude, (0.19,
> 0.27 s) for frequency, (0.13, 0.17 s) for AM, (0.07, 0.15 s) for FM and (0.04, 0.13 s) for
> higher-order properties.

One exponential with the constant fixed at the mean noise floor of each property, fitted from 0
to 4 s; the proportion at time t over all tests (ended ones as not significant, main) or over the
tests still running.

In [ ]:
paper8a = {"amplitude": (0.23, 0.25, 2.8e-3), "frequency": (0.19, 0.27, 2.3e-3), "AM": (0.13, 0.17, 2.3e-3), "FM": (0.07, 0.15, 2.1e-3),
           "higher-order": (0.04, 0.13, 3.6e-3)}
T4 = int(4.0 * cd.FS)
t4 = t_ax[:T4]
rows = []
for rd in READ8:
    S8 = sel8(rd)
    for denom in ("all", "running"):
        sizes = {}
        for p in PROPS:
            P_ = [t for t in S8 if t["prop"] == p]
            if len(P_) < 2:
                continue
            prop_, nf_, n_ = pop_curves(P_, denom)
            prop_, nf_ = prop_[:T4], nf_[:T4]
            kappa = float(np.nanmean(nf_))
            f = cd.fit_exp(t4, prop_, 1, kappa=kappa)
            a, tau = f["alpha"][0], f["tau"][0]
            sizes[p] = a
            rows.append(dict(reading=rd, denom=denom, prop=p, tests=len(P_), alpha1=a, tau1=tau, kappa=kappa, paper=paper8a[p],
                             alpha_ok=abs(a - paper8a[p][0]) <= 0.1 * paper8a[p][0], tau_ok=abs(tau - paper8a[p][1]) <= 0.1 * paper8a[p][1]))
        order = all(sizes.get(a, np.nan) > sizes.get(b, np.nan) for a, b in zip(PROPS[:-1], PROPS[1:]))
        rows.append(dict(reading=rd, denom=denom, prop="sizes in the order amplitude > frequency > AM > FM > higher-order", alpha1=float(order)))
F8F = pd.DataFrame(rows)
print(F8F.round(4).to_string())


**Result: not reproduced, resting on the property sets.** Main reading, 0 to 4 s (every test runs
at least 4 s, so the two denominators agree): sizes 0.145, 0.190, 0.087, 0.037 and 0.032 and
constants 0.246, 0.264, 0.178, 0.132 and 0.138 s for amplitude, frequency, AM, FM and higher-order,
with noise floors of 0.0022 to 0.0032. Four of the five constants are within tolerance (FM, 0.132 s,
is not) and one size (frequency). The sizes are not in the printed order, the amplitude size being
below the frequency size under every reading (0.145 against 0.190 in the main reading), so neither
the rule for reproduced nor that for partial holds.

## fig8b-decay-by-property

> The normalized context-dependent power decays slowly for amplitude and frequency changes (time
> constants 0.66 and 0.97 s) and fast for AM, FM and higher-order changes (0.11, 0.15 and 0.20 s),
> so higher-order properties act mainly on a timescale of about 100 ms.

Q(t) of Eq. 20 per property over its tests (as fig3b-fractional-power-decay), one exponential with
constant 0, fitted from 0 to 4 s (main) or to the shortest test of the property; the average at t
over the running tests or over all (ended ones as zero).

In [ ]:
paper8b = {"amplitude": (0.57, 0.66), "frequency": (0.54, 0.97), "AM": (0.86, 0.11), "FM": (0.64, 0.15), "higher-order": (0.52, 0.20)}


def q_curve(P_, denom, T=TT):
    st = {k: cd.stack_curves([t[k] for t in P_], T) for k in ("pred", "dep")}
    if denom == "running":
        av = {k: np.nanmean(v, 0) for k, v in st.items()}
    else:
        av = {k: np.nan_to_num(v).sum(0) / len(P_) for k, v in st.items()}
    return av["dep"] / cd.running_mean_0_2t(av["pred"])


rows = []
QP = {}
for rd in READ8:
    S8 = sel8(rd)
    for denom in ("running", "all"):
        taus = {}
        for p in PROPS:
            P_ = [t for t in S8 if t["prop"] == p]
            if len(P_) < 2:
                continue
            with np.errstate(invalid="ignore", divide="ignore"):
                Q = q_curve(P_, denom)
            QP[(rd, denom, p)] = Q
            short = min(t["dur"] for t in P_)
            for tmax in sorted({4.0, short}):
                m = (t_ax <= tmax) & np.isfinite(Q)
                f = cd.fit_exp(t_ax[m], Q[m], 1, kappa=0.0)
                if tmax == 4.0:
                    taus[p] = f["tau"][0]
                rows.append(dict(reading=rd, denom=denom, prop=p, tests=len(P_), t_max=round(tmax, 2), alpha1=f["alpha"][0], tau1=f["tau"][0],
                                 paper_alpha_tau=paper8b[p], tau_ok=abs(f["tau"][0] - paper8b[p][1]) <= 0.1 * paper8b[p][1],
                                 Q_first_50ms=float(np.nanmean(Q[:50])), Q_at_1s=float(Q[1000])))
        partial = (all(taus.get(p, 0) > 0.4 for p in ("amplitude", "frequency")) and all(taus.get(p, 9) < 0.3 for p in ("AM", "FM", "higher-order")))
        rows.append(dict(reading=rd, denom=denom, prop="partial pattern (amplitude, frequency > 0.4 s; AM, FM, higher-order < 0.3 s; 0-4 s)", alpha1=float(partial)))
F8B = pd.DataFrame(rows)
print(F8B.round(3).to_string())


**Result: not reproduced, resting on the property sets.** Main reading, 0 to 4 s, running tests:
tau1 is 0.349 s for amplitude, 0.731 s for frequency, 0.095 s for AM, 0.113 s for FM and 0.297 s for
higher-order (sizes 0.562, 0.515, 0.996, 0.711 and 0.905); none is within tolerance. AM, FM and
higher-order are below 0.3 s and frequency above 0.4 s, but amplitude is below 0.4 s, so the partial
pattern fails; the same holds over all tests (0.350 s) and with every set as a test (0.340 s). With
the naturalistic probes added, amplitude gives 1.39 or 1.56 s and frequency 0.87 or 0.94 s, and the
partial pattern holds. The outcome therefore depends on the reading; the main reading follows the
probe rows of Table 1, which list no naturalistic probes.

## fig8c-relative-effects

> Changes in amplitude and frequency of the conditioning stimuli produced context effects as large and
> long-lasting as natural sound contexts, while AM, FM and higher-order changes produced substantially
> smaller and shorter effects; A1 is more sensitive to lower-order than to higher-order properties.

Area under Q(t) from 0 to 4 s for each property over the area for the 305 natural sound probes
(fig3b-fractional-power-decay, silence counted as the main reference, not counted as a variant), with
95% intervals from 200 resamplings. Reading run: probes drawn with replacement, 1000 per resampling
(the paper's "1000 samples") and, as a variant, as many as the set holds. The trial reading (trials
drawn within each probe and context) is not run.

In [ ]:
T4 = int(4.0 * cd.FS)


def area_parts(P_):
    st = {k: cd.stack_curves([t[k] for t in P_], TT) for k in ("pred", "dep")}
    run = (~np.isnan(st["pred"])).astype(float)
    return np.nan_to_num(st["dep"]), np.nan_to_num(st["pred"]), run


def area_from(parts, w):
    dep, pred, run = parts
    n = w @ run
    with np.errstate(invalid="ignore", divide="ignore"):
        Q = (w @ dep / n) / cd.running_mean_0_2t(w @ pred / n)
    return np.nansum(Q[:T4]) / cd.FS


NATP = {inc: area_parts(NPf[inc]) for inc in (True, False)}
rows = []
for rd in READ8:
    S8 = sel8(rd)
    res = {}
    for p in PROPS:
        P_ = [t for t in S8 if t["prop"] == p]
        if len(P_) < 2:
            continue
        parts = area_parts(P_)
        for inc in (True, False):
            nat_area = area_from(NATP[inc], np.ones(len(NPf[inc])))
            point = area_from(parts, np.ones(len(P_))) / nat_area
            for size in ("1000", "n"):
                rng = seed("fig8c", rd, p, inc, size)
                k1 = 1000 if size == "1000" else len(P_)
                k2 = 1000 if size == "1000" else len(NPf[inc])
                bs = []
                for _ in range(200):
                    w1 = np.bincount(rng.integers(0, len(P_), k1), minlength=len(P_)).astype(float)
                    w2 = np.bincount(rng.integers(0, len(NPf[inc]), k2), minlength=len(NPf[inc])).astype(float)
                    bs.append(area_from(parts, w1) / area_from(NATP[inc], w2))
                lo, hi = np.percentile(bs, [2.5, 97.5])
                res[(p, inc, size)] = (point, lo, hi)
                rows.append(dict(reading=rd, reference_silence=inc, samples=size, prop=p, tests=len(P_), ratio=point, ci_lo=lo, ci_hi=hi))
    for inc in (True, False):
        for size in ("1000", "n"):
            r = {p: res.get((p, inc, size)) for p in PROPS}
            if any(v is None for v in r.values()):
                continue
            low_hi = min(r["amplitude"][1], r["frequency"][1])
            full = (all(r[p][2] >= 1 for p in ("amplitude", "frequency")) and all(r[p][2] < 1 and r[p][2] < low_hi for p in ("AM", "FM", "higher-order")))
            point = (all(r[p][0] > max(r[q][0] for q in ("AM", "FM", "higher-order")) for p in ("amplitude", "frequency")) and all(r[p][0] < 1 for p in ("AM", "FM", "higher-order")))
            rows.append(dict(reading=rd, reference_silence=inc, samples=size, prop="rule: intervals (reproduced) / point estimates (partial)", ratio=float(full), ci_lo=float(point)))
F8C = pd.DataFrame(rows)
print("natural sound area, 0 to 4 s: silence counted %.3f, not counted %.3f (s)" % (area_from(NATP[True], np.ones(len(NPf[True]))), area_from(NATP[False], np.ones(len(NPf[False])))))
print(F8C.round(3).to_string())


**Result: not reproduced, resting on the property sets.** Main reading, against the natural sounds
with silence counted: the area ratio is 0.393 for amplitude (95% interval 0.332 to 0.469 with 1000
probes per resampling, 0.232 to 0.596 with as many as the set holds), 0.616 for frequency (0.552 to
0.708; 0.462 to 0.856), 0.085 for AM, 0.203 for FM and 0.452 for higher-order. The amplitude
interval excludes 1 under every reference and sample size (highest bound 0.715), and the
higher-order estimate is above the amplitude one, so neither rule holds. With every set as a test,
or with the naturalistic probes added, the point estimates fall in the pattern (partial), and with
the naturalistic probes the amplitude and frequency intervals include 1 for the reference without
silence when the resampling draws as many probes as the set holds. The trial reading of the
resampling was not run.

## fig8-natural-synthetic-probes-alike

> Natural and synthetic probes gave no difference in the context effects of the synthetic
> conditioning stimuli, so they were combined.

Per property, the proportion of significant tests of natural and of synthetic probes compared by
Fisher's exact test at 0.05, Bonferroni over the properties tested with both probe types
(claims file); reproduced when no property shows a difference.

In [ ]:
rows = []
for rd in READ8:
    d = pd.DataFrame([dict(prop=t["prop"], natural=t["kind"] == "natural", cell=t["cell"], sig=bool(t["sig"].any())) for t in sel8(rd)])
    both = [p for p in PROPS if d[(d.prop == p)].natural.nunique() == 2]
    for p in PROPS:
        x = d[d.prop == p]
        a, b = int(x[x.natural].sig.sum()), int((~x[x.natural].sig).sum())
        c_, e = int(x[~x.natural].sig.sum()), int((~x[~x.natural].sig).sum())
        if p in both:
            orr, pv = stats.fisher_exact([[a, b], [c_, e]])
        else:
            orr, pv = np.nan, np.nan
        rows.append(dict(reading=rd, prop=p, natural=f"{a}/{a + b}", natural_cells=x[x.natural].cell.nunique(), synthetic=f"{c_}/{c_ + e}",
                         synthetic_cells=x[~x.natural].cell.nunique(), odds_ratio=orr, p=pv, p_bonferroni=min(1.0, pv * len(both)) if p in both else np.nan))
F8N = pd.DataFrame(rows)
print(F8N.round(4).to_string())
for rd in READ8:
    x = F8N[F8N.reading == rd]
    print(f"{rd}: properties tested with both probe types {int(x.p.notna().sum())}; with a difference after Bonferroni: {int((x.p_bonferroni < 0.05).sum())}")


**Result: reproduced, resting on the property sets and on the claims file's test.** In the main
reading, natural and synthetic probes give amplitude 10 of 20 against 33 of 50 (p = 0.28), frequency
12 of 21 against 40 of 64 (p = 0.80), AM 11 of 25 against 8 of 14 (p = 0.51) and higher-order 22 of 58
against 19 of 73 (p = 0.18); FM has no synthetic probes. No property differs after the Bonferroni
correction (smallest corrected p 0.74), and none does under the other readings. The synthetic AM
probes come from 2 cells, so that comparison has little power.

## figs1-silence-probe

> A subset of cells showed significant context dependence soon after the onset of the intersequence
> silences, but the response power attributable to the preceding stimulus alone faded within about
> 1 s after the stimulus ended.

The silence probe is the first 2.5 s after each sequence (CxtDep.m), its contexts the last fragment
of each sequence. Reproduced when at least one cell has a significant period starting within 0.5 s
of the silence onset, and the stimulus-related power (Eq. 17, averaged over silence probes) falls
below 10% of its onset value (mean of the first 10 ms) by 1.1 s. Eq. 19 is a variant.

In [ ]:
rows = []
for p in SILP:
    s = p["sig"]
    on = np.where(s)[0]
    rows.append(dict(cell=p["cell"], path=p["path"], nctx=len(p["ctx"]), rep=p["rep"], any_sig=bool(s.any()),
                     first_sig=on[0] / cd.FS if len(on) else np.nan))
SP = pd.DataFrame(rows)
SP = SP[SP.rep >= 4]
early = SP[SP.first_sig <= 0.5]
print(f"silence probes {len(SP)} in {SP.cell.nunique()} cells; with any significant period {int(SP.any_sig.sum())}; "
      f"starting within 0.5 s: {len(early)} probes in {early.cell.nunique()} cells")
TS2 = int(2.5 * cd.FS)
predS = np.nanmean(cd.stack_curves([p["pred"] for p in SILP if p["rep"] >= 4], TS2), 0)
depS = np.nanmean(cd.stack_curves([p["dep"] for p in SILP if p["rep"] >= 4], TS2), 0)
tS = np.arange(TS2) / cd.FS
for name, y in (("Eq. 17 (stimulus-related)", predS), ("Eq. 19 (context-dependent)", depS)):
    y0 = float(np.mean(y[:10]))
    print(f"{name}: onset {y0:.3f} mV^2, at 0.5 s {y[500]:.3f}, at 1.0 s {y[1000]:.3f}, at 2.0 s {y[2000]:.3f}; "
          f"first below 10% of onset at {cd.first_cross(tS, y, 0.1 * y0):.3f} s")
noise_sil = np.nanmean(cd.stack_curves([p["noise"] for p in SILP if p["rep"] >= 4], TS2), 0)
print(f"noise floor of the silence probes: mean {noise_sil.mean():.4f}")

**Result: reproduced.** Of 44 silence probes in 38 cells, 26 have a significant period, and 22 (in 19
cells) have one starting within 0.5 s of the silence onset. The stimulus-related power over the
silence probes falls from 5.61 mV^2 at onset to 1.34 at 0.5 s and below 10% of its onset value at
0.54 s (Eq. 19: 0.46 s). The noise floor of the silence probes is 0.0057.

## Claims not tested

- fig5-example-conductances, fig6-conductance-decay: every file of the data set is a current-clamp
  or cell-attached recording in mV (data check); the 42 voltage-clamp cells are not deposited.
- fig7-example-frequency-context: no recording has the design (see its section).


## Summary

| Claim | Outcome |
|---|---|
| methods-recording-counts | partially reproduced: natural sound (305 probes, 39 cells) and MGB (93, 14) samples exact; 132 A1 cells meet the rule against 123; no voltage-clamp data |
| a1-mgb-firing-rates | partially reproduced (assumption: A1 spike detector) |
| fig2-example-reliability | reproduced |
| fig2-example-long-context | reproduced (assumption: pairwise reliability): the example probe, identified by its reliability, is significant before 2 s, not from 1.6 to 3.97 s, and again after 3.97 s |
| fig3a-proportion-significant | reproduced |
| fig3a-significance-decay | partially reproduced |
| fig3b-fractional-power-decay | not reproduced: size matches, time constant 1.27 to 1.78 s against 1.04 s in every listed reading |
| timescale-longer-than-membrane | reproduced |
| fig4-predictability-upper-bound | not reproduced: bound near onset matches, time constant 1.45 to 2.81 s against 1.04 s |
| fig4-strf-performance-low | partially reproduced (`strf_asari_2009.ipynb`) |
| fig5-example-conductances, fig6-conductance-decay | blocked: no voltage-clamp recordings |
| fig8a-proportion-by-property | partially reproduced (assumption: property sets): order of the proportions holds, counts lower |
| fig8a-significance-fits | not reproduced (assumption: property sets): 4 of 5 constants match, sizes out of order |
| fig8b-decay-by-property | not reproduced (assumption: property sets): amplitude constant 0.35 s |
| fig8c-relative-effects | not reproduced (assumption: property sets): amplitude and frequency effects smaller than natural |
| fig8-natural-synthetic-probes-alike | reproduced (assumptions: property sets, the claims file's test) |
| fig9-mgb-fast-decay | reproduced |
| fig9-example-first-bin | reproduced |
| fig7-example-frequency-context | blocked: no recording with the example's design (all 105 synthetic recordings searched) |
| figs1-silence-probe | reproduced |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

How the analysis differs from the paper's. It runs at 1 kHz on the 10-ms median-filtered membrane
potential where the paper used 0.1 ms; on the probes of one recording both give the same
significant periods (agreement 0.997 to 1.000). Probe construction, the silence context and
probe, and the trial truncation follow the authors' code deposited with the data (CxtDep.m). A1
spikes are detected automatically, where the paper did it by hand. Fit ranges, the probes averaged at
each time, and the bin times of the MGB PSTH are run as readings. The sets of conditioning stimuli
for Figure 8 are rebuilt from the stimulus parameters, as the files name them only for six
recordings.

What was found about the dataset. It holds no voltage-clamp recording, so Figures 5 and 6 cannot
be tested. It holds 145 of the 194 A1 current-clamp cells and 14 of the 16 MGB cells, including all
the cells of the natural sound and MGB analyses. Which synthetic stimuli form the paper's property
groups is written in the notes of 6 of 105 synthetic recordings only, and few of the MHT probes of
Table 1 are in the data set (14 AM tests in 2 cells against 71 AM, 57 FM, 25 amplitude and 25
frequency probes). Two cells are listed in both
the natural and the synthetic group of Summary.mat. Four MGB cells come from an animal recorded as
29 days old, outside the paper's 20 to 28 days, and one cell has no age. The trigger times share the
clock of the traces (data check). The data set includes the authors' analysis code, which settles
several details the paper leaves open.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")